# Maintenance Log Query

Query maintenance records in natural language using local AI inference.

**Prerequisites:** Ollama running with `qwen2.5:3b` pulled. Install: `pip install requests pandas`

**Note:** Synthetic data. In production, connect to your actual CMMS.

In [ ]:
import pandas as pd
import requests
import json

OLLAMA_URL = 'http://localhost:11434/api/chat'
MODEL = 'qwen2.5:3b'

## Generate Synthetic Maintenance Records

In [ ]:
records = [
    {'date': '2025-04-12', 'equipment_id': 'T-101', 'type': 'Preventive', 'desc': 'Annual oil sampling and DGA. All values normal. Moisture at 12 ppm.', 'tech': 'J. Martinez', 'hrs': 3.0},
    {'date': '2025-05-08', 'equipment_id': 'BRK-205', 'type': 'Corrective', 'desc': 'Replaced SF6 pressure sensor (intermittent faults). P/N 4420-B. Tested OK.', 'tech': 'R. Chen', 'hrs': 4.5},
    {'date': '2025-06-21', 'equipment_id': 'T-101', 'type': 'Corrective', 'desc': 'Cooling fan #3 motor replaced. Bearing failure. Fan bank restored to full capacity.', 'tech': 'J. Martinez', 'hrs': 6.0},
    {'date': '2025-08-03', 'equipment_id': 'T-101', 'type': 'Preventive', 'desc': 'Thermal imaging survey. Hotspot on B-phase bushing connection. Torque check scheduled.', 'tech': 'R. Chen', 'hrs': 1.5},
    {'date': '2025-08-10', 'equipment_id': 'T-101', 'type': 'Corrective', 'desc': 'B-phase bushing re-torqued to 85 ft-lbs (was 62, min 75). Thermal re-scan confirmed resolved.', 'tech': 'J. Martinez', 'hrs': 2.5},
    {'date': '2025-10-22', 'equipment_id': 'T-101', 'type': 'Preventive', 'desc': 'Follow-up DGA. Acetylene at 3 ppm (prev <1). Trending but below 7 ppm action level. Next sample in 3 months.', 'tech': 'J. Martinez', 'hrs': 3.0},
    {'date': '2025-11-30', 'equipment_id': 'CT-401', 'type': 'Corrective', 'desc': 'Replaced secondary wiring (insulation degradation). Megger: 450 MOhm (min 100).', 'tech': 'R. Chen', 'hrs': 5.0},
    {'date': '2026-01-14', 'equipment_id': 'T-101', 'type': 'Preventive', 'desc': 'Quarterly DGA. Acetylene at 5 ppm (rising from 3). Hydrogen at 85 ppm. Recommend increased monitoring.', 'tech': 'J. Martinez', 'hrs': 3.0},
    {'date': '2026-03-01', 'equipment_id': 'T-101', 'type': 'Inspection', 'desc': 'Emergency DGA. Acetylene at 8 ppm (above 7 ppm action level). Recommend load reduction and engineering review.', 'tech': 'J. Martinez', 'hrs': 3.0},
]
df = pd.DataFrame(records)
df['date'] = pd.to_datetime(df['date'])
print(f'Loaded {len(df)} maintenance records')
df[['date', 'equipment_id', 'type', 'desc']]

## Query: What maintenance was done on T-101?

In [ ]:
t101 = df[df['equipment_id'] == 'T-101'].sort_values('date')
context = t101.to_string(index=False)

response = requests.post(OLLAMA_URL, json={
    'model': MODEL,
    'messages': [
        {'role': 'system', 'content': 'You are a power systems maintenance analyst. Be specific about dates, measurements, and trends.'},
        {'role': 'user', 'content': f'Maintenance records for T-101:\n\n{context}\n\nSummarize the work history and flag any concerning trends.'}
    ],
    'stream': False,
})
print(response.json()['message']['content'])

## Extract Structured Data from LLM

In [ ]:
structured_prompt = f'Based on these T-101 records:\n\n{context}\n\nExtract as JSON only:\n{{"total_work_orders": <int>, "total_hours": <float>, "corrective_count": <int>, "dga_acetylene_trend": [{{"date": "...", "ppm": <int>}}], "risk_level": "low|medium|high|critical", "recommended_action": "<one sentence>"}}'

response = requests.post(OLLAMA_URL, json={
    'model': MODEL,
    'messages': [{'role': 'user', 'content': structured_prompt}],
    'stream': False,
})
raw = response.json()['message']['content']
print('Raw response:')
print(raw)
print()
try:
    cleaned = raw.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.split('\n', 1)[1].rsplit('```', 1)[0]
    parsed = json.loads(cleaned)
    print('Parsed:')
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError:
    print('LLM response was not valid JSON. Retry or use a larger model.')

## Summary

Demonstrated natural language querying over maintenance records and structured data extraction. The rising acetylene trend in T-101 is the kind of insight AI can surface before it becomes a failure.